## I - Importing libraries + Defining parameters and data generation

In [49]:
import numpy as np
import random
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import plotly.express as px
from ortools.sat.python import cp_model
from IPython.display import display

class Config:
    T_BASE = 0
    P = 0
    Q = 0
    DELTA_PARALLEL = 0
    MULTIPLIERS = None

def initialize(t_base, p, q, delta_parallel):
    Config.T_BASE = t_base
    Config.P = p
    Config.Q = q
    Config.DELTA_PARALLEL = delta_parallel
    Config.MULTIPLIERS = {
        'I': 1, # Intermediate
        'A': p, # Advanced
        'B': q  # Beginner
    }

def generate_solvable_dependencies(num_tasks, num_cycles=2):
    """
    Generates dependencies formatted as a list of tuples:
    [('t1', 't2', {'relation': 'acyc'}), ('t2', 't3', {'relation': 'cyc'}), ...]
    """
    dependencies = []
    
    # 1. Build a strict feed-forward DAG backbone
    for i in range(2, num_tasks + 1):
        parent_idx = random.randint(1, i - 1)
        parent = f't{parent_idx}'
        child = f't{i}'
        dependencies.append((parent, child, {'relation': 'acyc'}))
        
    # 2. Inject multiple randomized, safe local cycles
    possible_cycle_starts = list(range(1, num_tasks))
    random.shuffle(possible_cycle_starts)
    
    cycles_added = 0
    for start_idx in possible_cycle_starts:
        if cycles_added >= num_cycles:
            break
            
        u = f't{start_idx}'
        v = f't{start_idx + 1}'
        
        edge_entry = (u, v, {'relation': 'cyc'})
        reverse_entry = (v, u, {'relation': 'cyc'})
        
        if edge_entry not in dependencies and reverse_entry not in dependencies:
            dependencies.append(edge_entry)
            cycles_added += 1
            
    return dependencies

def generate_random_availability(max_horizon, min_blocks, max_blocks):
    """
    Generates a list of random, non-overlapping intermittent availability windows.
    Example output: [(12, 180), (250, 410), (600, 850)]
    """
    windows = []
    # Determine how many availability blocks this developer will have
    num_blocks = random.randint(min_blocks, max_blocks)
    
    # Divide the timeline roughly into segments to prevent total overlap chaos
    segment_size = max_horizon // (num_blocks + 1)
    
    for i in range(num_blocks):
        # Define a safe lower and upper bound for this specific block
        start_min = i * segment_size + random.randint(0, 20)
        start_max = start_min + (segment_size // 2)
        
        start = random.randint(min(start_min, max_horizon - 50), min(start_max, max_horizon - 30))
        duration = random.randint(50, segment_size)
        finish = min(start + duration, max_horizon)
        
        if start < finish:
            windows.append((start, finish))
            
    # Sort windows sequentially just in case
    windows.sort(key=lambda x: x[0])
    return windows

def generate_solvable_project_data(num_tasks=10, num_devs=4, seed=42, max_attempts=100):
    """
    Generates a solvable, realistic project dataset with a valid DAG 
    and complete profile coverage.
    """
    random.seed(seed)
    
    # 1. Define standard types, modules, and team mappings
    all_types = ['database', 'backend', 'frontend', 'testing', 'devops']
    all_modules = ['data', 'auth', 'api', 'ui', 'dashboard', 'qa', 'infra']
    module_to_team = {
        'data': 'team_B', 
        'auth': 'team_A', 
        'api': 'team_A', 
        'ui': 'team_A', 
        'dashboard': 'team_A', 
        'qa': 'team_B', 
        'infra': 'team_B'
    }
    
    # 2. Generate Tasks
    task_ids = [f't{i}' for i in range(1, num_tasks + 1)]
    sp_list = [random.choice([2, 3, 4, 5, 8]) for _ in range(num_tasks)]
    type_list = [random.choice(all_types) for _ in range(num_tasks)]
    module_list = [random.choice(all_modules) for _ in range(num_tasks)]
    
    tasks = pd.DataFrame({
        'task_id': task_ids,
        'sp': sp_list,
        'type': type_list,
        'module': module_list
    }).set_index('task_id')
    
    tasks['team_required'] = tasks['module'].map(module_to_team)
    
    # 3. Generate Developers (Guaranteeing full profile coverage for all task modules/types)
    dev_ids = [f'd{i}' for i in range(1, num_devs + 1)]
    profiles = []
    teams = []
    exps = []
    availabilities = []

    # Ensure developers collectively cover every required type/module
    universal_profile = list(set(all_types + all_modules))
    for i in range(num_devs):
        if i == 0:
            # First dev is a senior full-stack expert covering everything to prevent bottlenecks
            profiles.append(universal_profile)
            teams.append('team_A')
            exps.append('A')
        else:
            profiles.append(random.sample(all_types, k=min(3, len(all_types))))
            teams.append(random.choice(['team_A', 'team_B']))
            exps.append(random.choice(['I', 'B']))
        availabilities.append(generate_random_availability(1000, 3, 4))
        
    developers = pd.DataFrame({
        'dev_id': dev_ids,
        'profile': profiles,
        'team': teams,
        'exp': exps,
        'availability': availabilities
    }).set_index('dev_id')
    
    # 4. Generate Safe, Strictly Feed-Forward Dependencies (Valid DAG)
    # This guarantees no deadlocks or impossible structural loops.
    dependencies = generate_solvable_dependencies(num_tasks)

    for attempt in range(max_attempts):
        random.seed(seed + attempt)
        # ... generate ...
        if _is_feasible(tasks, developers, dependencies):
            return tasks, developers, dependencies
        raise RuntimeError("Could not generate feasible instance")

def _is_feasible(tasks, developers, dependencies):
    result = cp_sat_scheduler(tasks, developers, dependencies)
    return bool(result)

## II - Building helper functions

### II.1 Dependency Graph

In [50]:
def build_dependency_graph(tasks, dependencies):
    G = nx.DiGraph()
    G_acyc = nx.DiGraph()

    task_data = tasks.to_dict('index')
    for task_id, data in task_data.items():
        G.add_node(task_id, **data)
        G_acyc.add_node(task_id, **data)

    for u, v, attr in dependencies:
        G.add_edge(u, v, **attr)
        if attr.get('relation') == 'acyc':
            G_acyc.add_edge(u, v)
        elif attr.get('relation') == 'cyc':
            G.add_edge(v, u, **attr)  # reciprocal for cycle detection only

    for node in G.nodes():
        G.nodes[node]['dep_count'] = len(nx.descendants(G_acyc, node))

    return G

### II.2 Feasibility Matrix

In [51]:
def check_feasibility(tasks, developers, task_id, dev_id):
    """
    Checks whether the developer (dev_id) is eligible for the giving task (task_id)
    """
    task = tasks.loc[task_id]
    dev = developers.loc[dev_id]
    
    # Rule 1: Skill Matching
    skill_match = task['type'] in dev['profile']

    # Rule 2: Team-Module Ownership
    team_match = dev['team'] == task['team_required']
    
    return skill_match and team_match

def build_feasibility_matrix(tasks, developers):
    matrix_data = []

    for task_id in tasks.index:
        row = []
        for dev_id in developers.index:
            if check_feasibility(tasks, developers, task_id, dev_id):
                row.append(1)
            else:
                row.append(0)
        matrix_data.append(row)

    return pd.DataFrame(
        matrix_data, 
        index=tasks.index, 
        columns=developers.index
    )

### II.3 Development Times

In [52]:
def a_dev_time(tasks, developers, task_id, dev_id):
    # dev_time(t_i, d_j) = sp(t_i) * T_{base} * beta(exp(d_j))
    task = tasks.loc[task_id]
    dev = developers.loc[dev_id]
    exp = dev['exp']

    return task['sp'] * Config.T_BASE * Config.MULTIPLIERS.get(exp)

def individual_workload(dev_id, tasks):
    return sum(a_dev_time(task_id, dev_id) for task_id in tasks.index)

def build_dev_times_matrix(tasks, developers):
    """
    Builds a matrix where rows are tasks and columns are developers,
    containing the calculated development time for each pair.
    """
    matrix_data = []

    for task_id in tasks.index:
        row = []
        for dev_id in developers.index:
            duration = a_dev_time(tasks, developers, task_id, dev_id)
            row.append(duration)
        matrix_data.append(row)

    return pd.DataFrame(
        matrix_data, 
        index=tasks.index, 
        columns=developers.index
    )

### II.4 Displayers

In [53]:
def display_feasibility_matrix(F):
    print("Feasibility Matrix:")
    display(F)
    
def display_dev_times_matrix(tasks, developers):
    dev_times = build_dev_times_matrix(tasks, developers)
    print("Development Time Matrix:")
    display(dev_times)

def display_availability_windows(developers):
    """
    Displays developer availability windows as a Gantt-style chart 
    matching your exact plotting and hover template style.
    """
    # 1. Unpack the availability intervals from the DataFrame into a flat plotting structure
    records = []
    for dev_id, row in developers.iterrows():
        avail_windows = row['availability']
        for i, (start, finish) in enumerate(avail_windows):
            records.append({
                'Developer': dev_id,
                'Window_ID': f"Window {i+1}",
                'Start': start,
                'Finish': finish
            })
            
    plot_df = pd.DataFrame(records)
    
    # If no availability data is present, handle gracefully
    if plot_df.empty:
        print("No availability windows to plot.")
        return

    plot_df['Duration'] = plot_df['Finish'] - plot_df['Start'] 
    
    # 2. Plot using Plotly Express with the requested style format
    fig = px.bar(
        plot_df, 
        x="Duration",
        y="Developer", 
        color="Developer", 
        orientation='h', 
        base="Start",
        custom_data=["Start", "Finish", "Duration"], 
        title="Developer Availability Windows Over Time"
    )
    
    # 3. Custom hover template matching your style structure
    fig.update_traces(
        hovertemplate=(
            "Developer: %{y}<br>"
            "Start: %{customdata[0]}<br>"
            "Finish: %{customdata[1]}<br>"
            "Duration: %{customdata[2]}<extra></extra>"
        )
    )
    
    fig.update_layout(xaxis_title="Time Units", yaxis_title="Developers")
    fig.update_yaxes(autorange="reversed")
    
    fig.show()

def display_dependency_graph(tasks, dependencies):
    """
    Displays tasks grouped into their respective modules, crisscrossed by depedency links.
    """
    G = nx.DiGraph()
    
    # Add nodes with module attributes
    for t_id, row in tasks.iterrows():
        G.add_node(t_id, module=row['module'], type=row['type'])
        
    # Add edges with relation attributes
    for u, v, data in dependencies:
        G.add_edge(u, v, relation=data.get('relation', 'acyc'))

    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 1. Group tasks by module
    modules = tasks['module'].unique()
    num_modules = len(modules)
    
    # Compute macro-centers for each module pod in a circle
    pod_centers = {}
    radius_macro = 4.5
    for i, mod in enumerate(modules):
        angle = 2 * np.pi * i / num_modules
        pod_centers[mod] = np.array([radius_macro * np.cos(angle), radius_macro * np.sin(angle)])
        
    # 2. Compute local micro-positions for tasks inside their respective pods uniformly
    pos = {}
    pod_radii = {}
    
    for mod in modules:
        mod_tasks = [t for t, data in G.nodes(data=True) if data['module'] == mod]
        num_mod_tasks = len(mod_tasks)
        
        # Give the pod a fixed radius big enough to hold its tasks cleanly
        # e.g., scale radius based on the number of tasks in this module
        base_radius = max(1.2, 0.5 * np.sqrt(num_mod_tasks) + 0.8)
        pod_radii[mod] = base_radius
        
        # Arrange tasks regularly in a circle (or concentric circles if too many) inside the pod
        for idx, t in enumerate(mod_tasks):
            if num_mod_tasks == 1:
                # Single task sits right at the pod center
                pos[t] = pod_centers[mod]
            else:
                # Distribute evenly along the perimeter of an inner sub-radius
                inner_radius = base_radius * 0.6
                angle = 2 * np.pi * idx / num_mod_tasks
                offset = np.array([inner_radius * np.cos(angle), inner_radius * np.sin(angle)])
                pos[t] = pod_centers[mod] + offset

    # 3. Draw Clean Circular Module Pod Backgrounds
    color_palette = ['#ff9999', '#99ff99', '#9999ff', '#ffcc99', '#ffff99']
    for i, mod in enumerate(modules):
        color = color_palette[i % len(color_palette)]
        
        # Using Circle patch with equal aspect ratio ensured by ax.set_aspect('equal')
        circle = patches.Circle(
            pod_centers[mod], pod_radii[mod], 
            color=color, alpha=0.3, zorder=0
        )
        ax.add_patch(circle)
        
        # Add Module Label Header at the top-center of each pod circle
        ax.text(
            pod_centers[mod][0], pod_centers[mod][1] + pod_radii[mod] - 0.3, 
            f"Module: {mod}", fontweight='bold', fontsize=10, 
            ha='center', va='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
        )

    # 4. Categorize Edges for Styling
    intra_edges = []
    inter_edges = []
    cyclic_edges = []
    
    cyclic_pairs = set()
    for u, v, data in G.edges(data=True):
        if data.get('relation') == 'cyc' or G.has_edge(v, u):
            cyclic_pairs.add(tuple(sorted((u, v))))

    for u, v, data in G.edges(data=True):
        u_mod = G.nodes[u]['module']
        v_mod = G.nodes[v]['module']
        
        if tuple(sorted((u, v))) in cyclic_pairs:
            cyclic_edges.append((u, v))
        elif u_mod == v_mod:
            intra_edges.append((u, v))
        else:
            inter_edges.append((u, v))

    # 5. Render Nodes & Labels
    nx.draw_networkx_nodes(G, pos, node_size=400, node_color='white', edgecolors='black', ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)

    # 6. Render Edges with Custom Curves & Colors
    # Intra-module (Gray, subtle inside pods)
    nx.draw_networkx_edges(G, pos, edgelist=intra_edges, edge_color='gray', arrows=True, width=1, ax=ax)
    
    # Inter-module (Green, curved to bypass pods cleanly)
    nx.draw_networkx_edges(
        G, pos, edgelist=inter_edges, edge_color='green', 
        arrows=True, width=1.5, connectionstyle='arc3,rad=0.2', ax=ax
    )
    
    # Cyclic Dependencies (Red, prominent curved arrows)
    nx.draw_networkx_edges(
        G, pos, edgelist=cyclic_edges, edge_color='red', 
        arrows=True, width=2.5, connectionstyle='arc3,rad=0.3', ax=ax
    )

    # Ensure circular pods do not distort into ovals when scaling canvas
    ax.set_aspect('equal')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    
def display_gantt_chart(schedule_df):
    if isinstance(schedule_df, dict):
        plot_df = pd.DataFrame.from_dict(schedule_df, orient='index', columns=['assigned_dev', 'start', 'finish'])
    else:
        plot_df = schedule_df.copy()

    plot_df = plot_df.reset_index()
    plot_df.columns = ['Task', 'Developer', 'Start', 'Finish']
    
    plot_df['Duration'] = plot_df['Finish'] - plot_df['Start'] 
    
    fig = px.bar(
        plot_df, 
        x="Duration",
        y="Task", 
        color="Developer", 
        orientation='h', 
        base="Start",
        # Include "Developer" in custom_data alongside Start, Finish, and Duration
        custom_data=["Developer", "Start", "Finish", "Duration"], 
        title="Project Schedule"
    )
    
    # Map each custom_data index correctly:
    # [0] = Developer, [1] = Start, [2] = Finish, [3] = Duration
    fig.update_traces(
        hovertemplate=(
            "Task: %{y}<br>"
            "Developer: %{customdata[0]}<br>"
            "Start: %{customdata[1]}<br>"
            "Finish: %{customdata[2]}<br>"
            "Duration: %{customdata[3]}<extra></extra>"
        )
    )
    
    fig.update_layout(xaxis_title="Time Units", yaxis_title="Tasks")
    fig.update_yaxes(autorange="reversed")
    
    fig.show()

## III - Objective Functions & Constraints

In [54]:
def calculate_coordination_risk(u, v, su, fu, sv, fv, G):
    """
    Calculates the Coordination Risk (CR) based on the scheduling relationship between task_u and task_v.
    """
    overlap = min(fu, fv) - max(su, sv)
    
    # Sequential condition
    if overlap < Config.DELTA_PARALLEL:
        return 0
    
    # Parallel condition for acyclic (cyclic risk is handled by the cycle loop beneath)
    return 1 + G.nodes[v].get('dep_count', 0)

def calculate_average_workload(x_ij, dev_times):
    """
    x_ij: DataFrame where rows are tasks, columns are developers (1 if assigned, 0 otherwise). \\
    dev_times: Matrix of pre-calculated completion times for each task/dev pair.
    """
    # Multiply assignment matrix by duration matrix
    individual_workloads = (x_ij * dev_times).sum(axis=0)
    
    # Calculate average W
    return individual_workloads.mean()

def calculate_makespan(schedule):
    # Makespan is simply the maximum finish time
    return schedule['finish'].max()

def calculate_workload_imbalance(schedule, developers):
    workloads = []
    for dev_id in developers.index:
        dev_tasks = schedule[schedule['assigned_dev'] == dev_id]
        w_j = (dev_tasks['finish'] - dev_tasks['start']).sum()
        workloads.append(w_j)
    return np.std(workloads)

def calculate_total_coordination_risk(G, schedule):
    total_risk = 0
    # Safely convert to a DataFrame if it's passed as a dictionary
    if isinstance(schedule, dict):
        schedule_df = pd.DataFrame.from_dict(schedule, orient='index', columns=['assigned_dev', 'start', 'finish'])
    elif isinstance(schedule, pd.DataFrame):
        schedule_df = schedule
    else:
        # Fallback conversion or handle array inputs safely
        schedule_df = pd.DataFrame(schedule, columns=['assigned_dev', 'start', 'finish'])
    
    # 1. Sum Acyclic Risks (P_acyc)
    # Filter edges that are NOT part of any cycle
    for u, v, data in G.edges(data=True):
        if data.get('relation') == 'acyc':
            su, fu = schedule_df.loc[u, 'start'], schedule_df.loc[u, 'finish']
            sv, fv = schedule_df.loc[v, 'start'], schedule_df.loc[v, 'finish']
            total_risk += calculate_coordination_risk(u, v, su, fu, sv, fv, G)

    # 2. Sum Cyclic Risks (P_cyc)
    # Build list of unique cyclic edge sets from dependencies
    cyc_edges = [(u, v) for u, v, attr in dependencies if attr.get('relation') == 'cyc']

    # Group into cycles (for your generator, each cyc edge is a 2-node cycle)
    seen_cycles = set()
    for u, v in cyc_edges:
        cycle_key = tuple(sorted((u, v)))
        if cycle_key in seen_cycles:
            continue
        seen_cycles.add(cycle_key)

        # Eq. (9): CRC = sum over edges in cycle of (1 + dep_count(t_k))
        cycle_risk = (1 + G.nodes[v].get('dep_count', 0)) + (1 + G.nodes[u].get('dep_count', 0))
        total_risk += cycle_risk
        
    return total_risk

def check_constraints(schedule, developers, dependencies):
    # 1. Skill & Team Constraints
    for task_id, row in schedule.iterrows():
        dev_id = row['assigned_dev']
        if not check_feasibility(tasks, developers, task_id, dev_id):
            print(f"Constraint Violated: Skill/Team mismatch for {task_id} assigned to {dev_id}.")
            return False

    # 2. Availability Constraint
    for task_id, row in schedule.iterrows():
        dev_id = row['assigned_dev']
        s_i, f_i = row['start'], row['finish']
        is_available = False
        for slot_start, slot_end in developers.loc[dev_id, 'availability']:
            if s_i >= slot_start and f_i <= slot_end:
                is_available = True
                break
        if not is_available:
            print(f"Constraint Violated: {task_id} interval [{s_i}, {f_i}] outside {dev_id} availability.")
            return False

    # 3. Flow Constraints (Acyclic & Cyclic)
    for u, v, attr in dependencies:
        su, fu = schedule.loc[u, 'start'], schedule.loc[u, 'finish']
        sv, fv = schedule.loc[v, 'start'], schedule.loc[v, 'finish']
        
        if attr['relation'] == 'acyc':
            if fv - su < Config.DELTA_PARALLEL:   # f_v - s_u >= delta
                print(f"Constraint Violated: Acyclic {u}->{v}: f_k - s_i = {fv - su} < {Config.DELTA_PARALLEL}")
                return False
        elif attr['relation'] == 'cyc':
            overlap = min(fu, fv) - max(su, sv)
            if overlap < Config.DELTA_PARALLEL:
                print(f"Constraint Violated: Cyclic dependency {u} <-> {v} lacks required overlap.")
                return False

    # 4. Resource Constraint (No overlapping tasks for the same dev)
    for dev_id in schedule['assigned_dev'].unique():
        dev_tasks = schedule[schedule['assigned_dev'] == dev_id].sort_values('start')
        for i in range(len(dev_tasks) - 1):
            if dev_tasks.iloc[i+1]['start'] < dev_tasks.iloc[i]['finish']:
                print(f"Constraint Violated: Resource conflict for {dev_id} on {dev_tasks.iloc[i].name} and {dev_tasks.iloc[i+1].name}.")
                return False 

    return True

## IV - Building the schedulers and the test-runner

In [ ]:
def cp_sat_scheduler(tasks, developers, dependencies):
    model = cp_model.CpModel()
    horizon = 1000 # Max project duration
    
    # 1. Variables
    starts = {t: model.NewIntVar(0, horizon, f'start_{t}') for t in tasks.index}
    ends = {t: model.NewIntVar(0, horizon, f'end_{t}') for t in tasks.index}
    # Binary variable: is task t assigned to dev d?
    assignments = {(t, d): model.NewBoolVar(f'assign_{t}_{d}') for t in tasks.index for d in developers.index}
    dev_time_matrix = build_dev_times_matrix(tasks, developers)
    G = build_dependency_graph(tasks, dependencies)
    
    # 2. Constraints
    # --- 1. TASK ASSIGNMENT & FEASIBILITY ---
    for t in tasks.index:
        # Each task must be assigned exactly once
        model.Add(sum(assignments[(t, d)] for d in developers.index) == 1)
        
        # Skill & Team constraints: forbid invalid assignments
        for d in developers.index:
            if not check_feasibility(tasks, developers, t, d):
                model.Add(assignments[(t, d)] == 0)

    # AVAILABILITY (Cleaned scope) ---
    for t in tasks.index:
        for d in developers.index:
            # Get intervals for this specific dev
            avail_intervals = developers.loc[d, 'availability']
            fits_in_window = []
            for i, (s_avail, e_avail) in enumerate(avail_intervals):
                fits = model.NewBoolVar(f'fits_{t}_{d}_{i}')
                model.Add(starts[t] >= s_avail).OnlyEnforceIf(fits)
                model.Add(ends[t] <= e_avail).OnlyEnforceIf(fits)
                fits_in_window.append(fits)
            
            # If assigned, it MUST fit
            model.Add(sum(fits_in_window) >= 1).OnlyEnforceIf(assignments[(t, d)])

    # --- 2. DURATION & TEMPORAL LINKING ---
    chosen_durations = {}

    for t in tasks.index:
        # Get the list of times
        dev_times = [int(dev_time_matrix.loc[t, d]) for d in developers.index]
        
        # Create the duration variable and store it in our dictionary
        chosen_durations[t] = model.NewIntVar(0, horizon, f'dur_{t}')
        
        # Link chosen developer to the corresponding duration
        for t in tasks.index:
            dev_times = [int(dev_time_matrix.loc[t, d]) for d in developers.index]
            chosen_durations[t] = model.NewIntVar(0, horizon, f'dur_{t}')
            for i, d in enumerate(developers.index):
                model.Add(chosen_durations[t] == dev_times[i]).OnlyEnforceIf(assignments[(t, d)])
            model.Add(ends[t] == starts[t] + chosen_durations[t])
        
        # Link start, end, and duration
        model.Add(ends[t] == starts[t] + chosen_durations[t])

    # --- 3. PROJECT DEPENDENCIES ---
    for u, v, attr in dependencies:
        if attr['relation'] == 'acyc':
            model.Add(ends[v] - starts[u] >= Config.DELTA_PARALLEL)
            
        elif attr['relation'] == 'cyc':
            # Logic: Enforce minimum overlap (DELTA_PARALLEL)
            overlap_start = model.NewIntVar(0, horizon, f'overlap_start_{u}_{v}')
            overlap_end = model.NewIntVar(0, horizon, f'overlap_end_{u}_{v}')
            
            model.AddMaxEquality(overlap_start, [starts[u], starts[v]])
            model.AddMinEquality(overlap_end, [ends[u], ends[v]])
            
            model.Add(overlap_end - overlap_start >= Config.DELTA_PARALLEL)

    # --- 4. RESOURCE CAPACITY (NO OVERLAP) ---
    for d in developers.index:
        # Create interval list per developer for resource scheduling
        intervals = [
            model.NewOptionalIntervalVar(
                starts[t], int(a_dev_time(tasks, developers, t, d)), ends[t], 
                assignments[(t, d)], f'opt_int_{t}_{d}'
            ) for t in tasks.index
        ]
        # Ensure one developer cannot work on two tasks at once
        model.AddNoOverlap(intervals)

    # 1. Makespan (The time the last task finishes)
    makespan = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(makespan, [ends[t] for t in tasks.index])

    # 2. Average Workload
    # Note: CP-SAT doesn't do floating point division well. 
    # Use total sum to represent "average".
    total_workload = model.NewIntVar(0, horizon * len(tasks), 'total_workload')
    model.Add(total_workload == sum(chosen_durations[t] for t in tasks.index))

    # 3. Workload Imbalance (Range as a proxy for standard deviation)
    max_dev_load = model.NewIntVar(0, horizon, 'max_load')
    min_dev_load = model.NewIntVar(0, horizon, 'min_load')
    dev_loads = []
    for d in developers.index:
        load = model.NewIntVar(0, horizon, f'load_{d}')
        model.Add(load == sum(assignments[(t, d)] * int(a_dev_time(tasks, developers, t, d)) for t in tasks.index))
        dev_loads.append(load)

    model.AddMaxEquality(max_dev_load, dev_loads)
    model.AddMinEquality(min_dev_load, dev_loads)
    imbalance = model.NewIntVar(0, horizon, 'imbalance')
    model.Add(imbalance == max_dev_load - min_dev_load)

    # 4. Coordination Risk ---
    risk_terms = []
    
    for u, v, attr in dependencies:
        # Create a boolean variable indicating whether tasks u and v overlap by at least DELTA_PARALLEL
        is_parallel = model.NewBoolVar(f'parallel_{u}_{v}')
        
        overlap_start = model.NewIntVar(0, horizon, f'overlap_start_{u}_{v}')
        overlap_end = model.NewIntVar(0, horizon, f'overlap_end_{u}_{v}')
        
        model.AddMaxEquality(overlap_start, [starts[u], starts[v]])
        model.AddMinEquality(overlap_end, [ends[u], ends[v]])
        
        # If (overlap_end - overlap_start) >= DELTA_PARALLEL, then is_parallel must be True (1)
        # We use indicator constraints or linear inequalities with Big-M/reification
        # CP-SAT allows linking boolean variables to linear comparisons:
        model.Add(overlap_end - overlap_start >= Config.DELTA_PARALLEL).OnlyEnforceIf(is_parallel)
        model.Add(overlap_end - overlap_start < Config.DELTA_PARALLEL).OnlyEnforceIf(is_parallel.Not())
        
        # If it's an acyclic dependency and they run in parallel (overlap >= delta), 
        # add the risk weight (e.g., factor including downstream descendants count)
        if attr['relation'] == 'acyc':
            descendant_count = len(list(nx.descendants(G, v)))
            risk_weight = 1 + descendant_count
            
            # Create an integer term for this edge's risk contribution
            edge_risk = model.NewIntVar(0, horizon, f'risk_{u}_{v}')
            model.AddMultiplicationEquality(edge_risk, [is_parallel, risk_weight])
            risk_terms.append(edge_risk)

        # Cyclic risk (Eq. 9): fixed penalty per cycle edge once cycle constraint is satisfied
        cyc_edges = [(u, v) for u, v, attr in dependencies if attr.get('relation') == 'cyc']
        seen = set()
        for u, v in cyc_edges:
            key = tuple(sorted((u, v)))
            if key in seen:
                continue
            seen.add(key)
            risk_terms.append(1 + G.nodes[v].get('dep_count', 0))
            risk_terms.append(1 + G.nodes[u].get('dep_count', 0))

            total_risk = model.NewIntVar(0, horizon * len(dependencies), 'total_risk')
            if risk_terms:
                model.Add(total_risk == sum(risk_terms))
            else:
                model.Add(total_risk == 0)

    W_MAKESPAN = 1
    W_IMBALANCE = 1
    W_RISK = 1

    model.minimize(W_MAKESPAN * makespan + W_IMBALANCE * imbalance + W_RISK * total_risk)

    # 3. Solve
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        result = {}
        for t in tasks.index:
            # Find the assigned developer (the one where the bool is 1)
            assigned_dev = next(d for d in developers.index if solver.Value(assignments[(t, d)]) == 1)
            
            # Ensure the value is a LIST or TUPLE of 3 elements
            result[t] = [
                assigned_dev, 
                solver.Value(starts[t]), 
                solver.Value(ends[t])
            ]
        print(f"{status} solution found.")
        return result
    else:
        print("No feasible solution found.")
        return {} # Return an empty dict if the solver failed

class HybridSchedulerPSOGA:
    def __init__(self, tasks, developers, dependencies, pop_size, max_iter):
        self.tasks = tasks
        self.developers = developers
        self.G = build_dependency_graph(tasks, dependencies)
        self.dev_times = build_dev_times_matrix(tasks, developers)
        self.pop_size = pop_size
        self.max_iter = max_iter
        self.task_list = list(tasks.index)
        self.dev_list = list(developers.index)
        
        self.global_best_position = None
        self.global_best_score = float('inf')

    #def find_valid_start_time(self, assigned_dev, duration):

    #def decode_particle(self, position):

    def create_random_particle(self):
        particle = []
        for task_id in self.task_list:
            compatible_devs = [d for d in self.dev_list if check_feasibility(tasks, developers, task_id, d)]
            if not compatible_devs:
                compatible_devs = self.dev_list
            assigned_dev = random.choice(compatible_devs)
            particle.append((task_id, assigned_dev))
        random.shuffle(particle)
        return particle

    def initialize_population(self):
        population = []
        for _ in range(self.pop_size):
            particle = self.create_random_particle()
            score, _ = self.decode_particle(particle)
            population.append({
                'position': particle,
                'fitness': score,
                'best_position': list(particle),
                'best_fitness': score
            })
            if score < self.global_best_score:
                self.global_best_score = score
                self.global_best_position = list(particle)
        return population

    def crossover(self, parent1, parent2):
        size = len(parent1)
        if size <= 1:
            return list(parent1)
        start, end = sorted(random.sample(range(size), 2))
        child = [None] * size
        child[start:end+1] = parent1[start:end+1]
        child_tasks = {item[0] for item in child if item is not None}
        
        pointer = 0
        for gene in parent2:
            if gene[0] not in child_tasks:
                while child[pointer] is not None:
                    pointer += 1
                child[pointer] = gene
                child_tasks.add(gene[0])
        return child

    def mutate(self, position):
        child = list(position)
        if random.random() < 0.4:
            if len(child) > 1:
                idx1, idx2 = random.sample(range(len(child)), 2)
                child[idx1], child[idx2] = child[idx2], child[idx1]
            idx = random.randint(0, len(child) - 1)
            task_id, _ = child[idx]
            compatible_devs = [d for d in self.dev_list if check_feasibility(tasks, developers, task_id, d)]
            if compatible_devs:
                new_dev = random.choice(compatible_devs)
                child[idx] = (task_id, new_dev)
        return child

    def optimize(self):
        population = self.initialize_population()
        for _ in range(self.max_iter):
            for particle in population:
                temp_pos = self.crossover(particle['position'], particle['best_position'])
                if random.random() < 0.5 and self.global_best_position:
                    temp_pos = self.crossover(temp_pos, self.global_best_position)
                
                new_position = self.mutate(temp_pos)
                score, _ = self.decode_particle(new_position)
                
                if score < particle['best_fitness']:
                    particle['best_fitness'] = score
                    particle['best_position'] = list(new_position)
                    
                    if score < self.global_best_score:
                        self.global_best_score = score
                        self.global_best_position = list(new_position)
                
                particle['position'] = new_position
                particle['fitness'] = score
                
        _, best_schedule = self.decode_particle(self.global_best_position)
        return best_schedule

def run_and_evaluate_scheduler(solver_name, tasks, developers, dependencies, pop_size=None, max_iter=None):
    """
    Unified execution function for different schedulers (CP-SAT or PSO-GA).
    Returns the schedule DataFrame and prints analysis metrics.
    """
    
    # 1. Execute the chosen solver
    if solver_name.lower() in ['cp-sat', 'cpsat', 'solver']:
        raw_schedule = cp_sat_scheduler(tasks, developers, dependencies)
    elif solver_name.lower() in ['pso-ga', 'hybrid', 'metaheuristic']:
        scheduler = HybridSchedulerPSOGA(tasks, developers, dependencies, pop_size, max_iter)
        raw_schedule = scheduler.optimize()
        print("Optimization Finished Successfully!")
    else:
        raise ValueError(f"Unknown solver name: {solver_name}. Choose 'cp-sat' or 'pso-ga'.")

    if not raw_schedule:
        print(f"[{solver_name}] Failed to generate a schedule.")
        return None

    # 2. Convert to DataFrame
    schedule_df = pd.DataFrame.from_dict(
        raw_schedule,
        orient='index',
        columns=['assigned_dev', 'start', 'finish']
    )
    
    print(f"\n--- Schedule Result ({solver_name.upper()}) ---")
    print(schedule_df)
    display_gantt_chart(schedule_df)

    # 3. Calculate Metrics
    makespan = calculate_makespan(schedule_df)
    print(f"Makespan: {makespan}")

    total_risk = calculate_total_coordination_risk(build_dependency_graph(tasks, dependencies), schedule_df)
    print(f"Total Coordination Risk: {total_risk}")

    imbalance = calculate_workload_imbalance(schedule_df, developers)
    print(f"Workload Imbalance: {imbalance}")

    is_valid = check_constraints(schedule_df, developers, dependencies)
    print(f"Is schedule valid? {is_valid}\n")

    return schedule_df

## V - Running the schedulers & Displaying the results

In [ ]:
initialize(10, 0.8, 1.2, 2)
tasks, developers, dependencies = generate_solvable_project_data(num_tasks=10, num_devs=3, max_attempts=10)
display_dependency_graph(tasks, dependencies)
display_dev_times_matrix(tasks, developers)
display_availability_windows(developers)
schedule_cpsat = run_and_evaluate_scheduler('cp-sat', tasks, developers, dependencies)
#schedule_psoga = run_and_evaluate_scheduler('pso-ga', tasks, developers, dependencies, pop_size=20, max_iter=30)